# 7.2 Checklist text table

Checklist table with each q's full text. Columns: Question, Category, BL Share, EL Share, and Change - the share of labs with "Yes" at baseline (and, separately, at endline) out of labs with a substantive answer ("Yes"/"No"/"I don't know") at that wave, i.e. N/A and unanswered are excluded from each wave's own denominator.

With 49 full-text questions the table is too long for one page, so it's built as a LaTeX longtable (not tabular) that breaks across pages. This means in the paper it must be `\input` directly rather than wrapped in `\begin{table}...\end{table}`/`\scalebox{}`. It also needs `\usepackage{longtable}` added to the document preamble.

Sample: treated labs with *both* BL and EL data.

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

sys.path.append(str(Path.cwd().parents[0] / "functions"))
from make_checklist_text_table import load_checklist_question_text, make_checklist_text_table

In [2]:
# Load data (blank cells are genuinely missing, not the string "NA")
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,
    na_values=[""]
)

## (1) Prepare data

In [3]:
# Treated labs with both BL and EL data - needed since this table now reports
# achievement share at both waves (a BL-only table wouldn't need this restriction).
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
matched_labgroupids = labgroup_counts[labgroup_counts == 2].index

df_bl = df[df["labgroupid"].isin(matched_labgroupids) & (df["survey"] == "BL")].copy()
df_treatment = df_bl[df_bl["treated"] == 1].copy()

print(f"Treated labs with both BL and EL data: {len(df_treatment)}")

Treated labs with both BL and EL data: 56


In [4]:
# Item order and section headers (same as 7.1)
items = (
    [f"bronze_q_{i}" for i in range(1, 17)]
    + [f"silver_q_{i}" for i in range(1, 19)]
    + [f"gold_q_{i}" for i in range(1, 16)]
)

section_headers = {
    "Bronze Checklist": [f"bronze_q_{i}" for i in range(1, 17)],
    "Silver Checklist": [f"silver_q_{i}" for i in range(1, 19)],
    "Gold Checklist": [f"gold_q_{i}" for i in range(1, 16)],
}

# Question categories (same as 7.1)
item_categories = {}
# Bronze qs
for i in range(1, 7):
    item_categories[f"bronze_q_{i}"] = "General Lab"
for i in range(7, 10):
    item_categories[f"bronze_q_{i}"] = "Offices \& Travel"
item_categories["bronze_q_10"] = "Cold Storage"
for i in range(11, 16):
    item_categories[f"bronze_q_{i}"] = "Chemistry"
item_categories["bronze_q_16"] = "Fume Cupboards"
# Silver qs
for i in range(1, 9):
    item_categories[f"silver_q_{i}"] = "General Lab"
for i in range(9, 12):
    item_categories[f"silver_q_{i}"] = "Offices \& Travel"
item_categories["silver_q_12"] = "Cold Storage"
for i in range(13, 18):
    item_categories[f"silver_q_{i}"] = "Chemistry"
item_categories["silver_q_18"] = "Fume Cupboards"
# Gold qs
for i in range(1, 7):
    item_categories[f"gold_q_{i}"] = "General Lab"
for i in range(7, 10):
    item_categories[f"gold_q_{i}"] = "Offices \& Travel"
item_categories["gold_q_10"] = "Cold Storage"
for i in range(11, 15):
    item_categories[f"gold_q_{i}"] = "Chemistry"
item_categories["gold_q_15"] = "Fume Cupboards"

In [5]:
# Full question text, keyed the same way as items (e.g. "bronze_q_1")
question_text = load_checklist_question_text(
    config.SURVEY_DICTIONARIES / "helper_survey_dictionary.xlsx",
    config.EL_TREATMENT_QUESTIONNAIRE,
)

/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## (2) Compute baseline and endline share of "Yes" among substantive answers

In [6]:
substantive = ("Yes", "No", "I don't know")

def compute_shares(suffix):
    shares = {}
    for item in items:
        resp = df_treatment[f"{item}_{suffix}"]
        is_substantive = resp.isin(substantive)
        n_substantive = is_substantive.sum()
        shares[item] = (resp == "Yes").sum() / n_substantive * 100 if n_substantive > 0 else float("nan")
    return shares

shares_bl = compute_shares("bl")
shares_el = compute_shares("el")

## (3) Build and save table

In [7]:
table = make_checklist_text_table(
    items, question_text, item_categories, shares_bl, shares_el, section_headers=section_headers,
    caption="Checklist actions",
    label="tab:checklist_actions",
)

out_dir = config.OUTPUT / "9_Checklist_Tables"
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "checklist_text_table.tex"
_ = table_path.write_text(table)